# Análise do Funil Comercial — Helios GD (fictícia)

**Objetivo:** responder, com dados, três perguntas de negócio:
1. Onde o funil perde mais negociações?
2. Quais canais de aquisição convertem melhor?
3. A base é confiável o suficiente para suportar essas conclusões?

> Dados 100% sintéticos, gerados por `data/generate_synthetic.py`.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from funnel import metrics as m, quality as q
pd.set_option('display.float_format', '{:.2f}'.format)

raw = m.load('../data/synthetic_deals.csv')
print(f'{len(raw):,} linhas | {raw.deal_id.nunique():,} negociações únicas')
raw.head()

5,050 linhas | 5,000 negociações únicas


,deal_id,cliente_id,canal,segmento,distribuidora,etapa,status,status_credito,kwh_declarado,kwh_calculado,desconto_pct,fidelidade_meses,motivo_perda,criado_em,fechado_em,contrato_assinado
0,D100000,C17167,Indicação,Residencial,Copel,Ganho,Ganho,Aprovado,712.00,700.00,15,24,NaN,2025-06-05,2025-07-23,Sim
1,D100001,C37765,Parceiro,Comercial,Copel,Qualificação,Perdido,NaN,1560.00,1423.00,12,0,Consumo abaixo do mínimo,2025-08-21,2025-09-09,Não
2,D100002,C96472,Outbound,Comercial,CPFL Paulista,Proposta,Perdido,NaN,1055.00,972.00,20,24,Optou por concorrente,2026-07-30,2026-08-20,Não
3,D100003,C48521,Indicação,Residencial,CEMIG,Proposta,Perdido,NaN,737.00,880.00,12,12,Desconto insuficiente,2025-03-10,2025-04-09,Não
4,D100004,C34692,Inbound,Residencial,CPFL Paulista,Ganho,Ganho,Aprovado,676.00,689.00,25,12,NaN,2025-03-14,2025-04-08,Sim


## 1. Qualidade de dados primeiro
Antes de qualquer KPI, aplico os testes de controle sobre **100% da população** — mesma lógica de um teste de auditoria.

In [2]:
q.run_checks(raw)

,codigo,teste,severidade,populacao,excecoes,pct,resultado
0,QD-01,Negociação duplicada (deal_id repetido),Alta,5050,50,0.01,Exceção
1,QD-02,Distribuidora não informada,Média,5050,103,0.02,Exceção
2,QD-03,kWh calculado ausente,Média,5050,76,0.02,Exceção
3,QD-04,Divergência kWh declarado × calculado > 30%,Alta,5050,150,0.03,Exceção
4,QD-05,Venda ganha sem contrato assinado,Alta,5050,31,0.01,Exceção
5,QD-06,Data de fechamento anterior à criação,Alta,5050,24,0.00,Exceção
6,QD-07,Perda sem motivo registrado,Baixa,5050,99,0.02,Exceção
7,QD-08,Etapa de contrato/ganho sem crédito aprovado,Alta,5050,17,0.00,Exceção


Exceções de severidade **Alta** afetam diretamente receita e compliance (ex.: venda ganha sem contrato). A análise segue com a base deduplicada; as exceções viram plano de ação para o time comercial.

In [3]:
df = m.dedupe(raw)
q.exceptions(raw, 'QD-05').head()

,deal_id,cliente_id,canal,segmento,distribuidora,etapa,status,status_credito,kwh_declarado,kwh_calculado,desconto_pct,fidelidade_meses,motivo_perda,criado_em,fechado_em,contrato_assinado
185,D100185,C62739,Indicação,Comercial,CEMIG,Ganho,Ganho,Aprovado,1948.00,2040.00,15,12,NaN,2025-05-23,2025-07-14,Não
611,D100611,C43043,Indicação,Comercial,CEMIG,Ganho,Ganho,Aprovado,3095.00,3168.00,15,36,NaN,2026-03-26,2026-04-30,Não
1027,D101027,C67485,Parceiro,Residencial,Celesc,Ganho,Ganho,Aprovado,337.00,539.00,12,36,NaN,2025-09-08,2025-10-03,Não
1042,D101042,C16746,Indicação,Comercial,Copel,Ganho,Ganho,Aprovado,2266.00,2445.00,15,24,NaN,2026-01-12,2026-02-23,Não
1052,D101052,C81947,Inbound,Comercial,Celesc,Ganho,Ganho,Aprovado,4274.00,4530.00,10,0,NaN,2025-07-28,2025-09-11,Não


## 2. Funil

In [4]:
m.funnel(df)

,etapa,negociacoes,conv_etapa_anterior,conv_acumulada
0,Novo,5000,NaN,1.00
1,Qualificação,4045,0.81,0.81
2,Proposta,2622,0.65,0.52
3,Análise de Crédito,1986,0.76,0.40
4,Contrato,1693,0.85,0.34
5,Ganho,1563,0.92,0.31


A maior queda ocorre entre **Qualificação → Proposta**: é a etapa que mais merece investigação (critério de qualificação, tempo de resposta, competitividade do desconto).

In [5]:
m.loss_reasons(df)

motivo_perda
Sem retorno do lead             881
Sem interesse                   639
Fora da área de concessão       460
Consumo abaixo do mínimo        433
Crédito reprovado               279
Desconto insuficiente           195
Optou por concorrente           193
(não informado)                  99
Desistiu antes da assinatura     65
Divergência contratual           57
Name: count, dtype: int64

## 3. Canais de aquisição

In [6]:
m.conversion_by(df, 'canal')

,encerradas,ganhas,taxa_ganho
canal,,,
Indicação,715,440,0.62
Parceiro,1701,642,0.38
Inbound,1498,384,0.26
Outbound,950,97,0.10


In [7]:
m.conversion_by(df, 'segmento')

,encerradas,ganhas,taxa_ganho
segmento,,,
Comercial,1467,499,0.34
Industrial,479,153,0.32
Residencial,2918,911,0.31


## 4. Ciclo de venda

In [8]:
m.cycle_time(df).describe()

count   1556.00
mean      37.54
std       13.78
min        1.00
25%       28.00
50%       35.00
75%       45.00
max      104.00
dtype: float64

## Conclusões

| # | Achado | Implicação |
|---|---|---|
| 1 | Indicação converte ~2x a média | Programa estruturado de indicação tende a ter o melhor retorno |
| 2 | Outbound converte ~10% | Rever ICP e roteiro antes de ampliar investimento |
| 3 | Maior perda em Qualificação → Proposta | Revisar critérios de qualificação e SLA de envio de proposta |
| 4 | 8 testes de qualidade com exceções | Controles preventivos no CRM (campos obrigatórios, travas de etapa) |
